### 🚀 Project: KoChatGPT 성능 업그레이드

* 기존의 KoChatGPT 모델 분석, 
* RLHF(SFT-RM-PPO) 파이프라인의 각 단계별 파라미터 최적화와 체계적인 평가 환경을 구축



#### step 1. 코드 리팩토링

* **주요 작업:** 노드(NODE) 코드의 모듈화 및 파라미터화.

#### step 2. 성능 평가 환경 준비

* **평가 도구:** `lm-evaluation-harness` 도입.

#### step 3. Baseline 설정

* **성능 확인:** 최적화 기법 적용 전, 순정 상태의 KoChatGPT 성능을 측정하여 개선 정도를 비교할 수 있는 대조군(Baseline) 데이터 확보.

#### step 4. RLHF 최적화: 각종 성능 향상 기법 적용 ( ing ... )

학습의 각 단계(Step)별로 주요 파라미터를 조정하며 성능 변화를 관찰.

1. **SFT (Supervised Fine-Tuning):**
* Learning Rate Scheduler 변경, Batch Size 조정, Prompt 템플릿 최적화.


2. **RM (Reward Model):**
* Ranking Loss의 Margin 값 조정, 데이터 셔플링 및 데이터 증강을 통한 심판 모델의 정교화.


3. **PPO (Proximal Policy Optimization):**
* KL Divergence Coefficient ($\beta$) 튜닝 (모델의 붕괴 방지), 에피소드 길이 설정 및 최적의 보상(Reward) 스케일 탐색.


#### 5. 프로젝트 회고 및 인사이트
* `.ipynb`(실험 및 시각화)와 `.py`(자동화 및 대규모 학습)를 병행 사용 -> 효율 상승


##### Setup

In [ ]:
# ------ colab ---------------------------------
# from google.colab import drive
# drive.mount("/content/drive")

# import os
# os.getcwd()

# os.chdir('/content/drive/MyDrive/projects/AIFFEL_quest_eng/NLP/NLP05')

# ------ node ---------------------------------
# ! pip install transformers loralib lm-eval 'accelerate>=1.1.0'


In [2]:
# %reload_ext autoreload
# %autoreload 2

import os
import json
import logging
import copy
from copy import deepcopy
import random
import functools
from typing import Optional, Dict, Sequence, List
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import Dataset
import pandas as pd
import numpy as np

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedTokenizerFast,
    GPT2Config,
    GPT2Model,
    pipeline
)

from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

from guide import * 

#### BaseLine Train

In [3]:
    """ NODE baseline: 그냥 찍는 수준 """

    model_name = "skt/kogpt2-base-v2"                       # V
    tcname = model_name.replace("/", "-") + "_baseline"

    try:
        root_path = os.path.dirname(os.path.abspath(__file__))
    except:
        root_path = os.getcwd()
        
    cfg = {
        "model_name": model_name,                       # V
        "sft_tokenizer": model_name,
        "device": torch.device("xpu" if torch.xpu.is_available() else "cuda" if torch.cuda.is_available() else "cpu"),
        "root_path": root_path,

        "sft_output_dir": os.path.join(root_path, "test", tcname),
        "sft_saved_dir": os.path.join(root_path, "models", tcname + "_output_1_SFT"),
        "rm_saved_dir": os.path.join(root_path, "models", tcname + "_output_2_RM"),
        "ppo_saved_dir": os.path.join(root_path, "models", tcname + "_output_3_PPO"),

        "data_path_1_SFT": os.path.join(root_path, "KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl"),
        "data_path_2_RM": os.path.join(root_path, "KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl"),
        "data_path_3_PPO": os.path.join(root_path, "KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl"),

        "sft_num_train_epochs": 1,              # V
        "sft_per_device_train_batch_size": 4,   # V
        "sft_per_device_eval_batch_size": 4,    # v 
        "sft_warmup_steps": 5,                  # V 해당 step 동안 학습률을 0 -> 목표값까지 증가
        "sft_prediction_loss_only": True,       # V 학습/평가 시 손실값(Loss)만 계산할지 예측값(Logits/Metrics)까지 계산할지 - 일단 Fix 
        "sft_fp16": False,                      # 메모리 절약

        "rm_batch_size": 4,                     # V
        "rm_max_epochs": 1,                     # V

         "rm_seed": 230319, 
         "rm_max_train_samples": 1000, 
         "rm_max_len": 512, 
         "rm_lr": 5e-5 
    }

    model = AutoModelForCausalLM.from_pretrained(cfg["model_name"]).to(cfg["device"])
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        cfg["model_name"],
        bos_token='</s>', eos_token='</s>', unk_token='<unk>', pad_token='<pad>', mask_token='<mask>',
        padding_side="right",
        model_max_length=512,
    )

    show_info(cfg)
    run_sft(cfg, model, tokenizer)
    run_reward_model(cfg, tokenizer)
    run_ppo(cfg, model, tokenizer)
    run_evaluation(cfg)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

...

>>> PPO training skipped as pre-trained model was loaded.

-------------------- generate_custom() --------------------
>>> Instruction:
불고기용 고기 한우에요?
>>> Response:\n\n"I hope more about this squory about information.", 'token': 84} 生育生官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官官
>>> Instruction:
리처드 닉슨이 43대 부통령직을 수행한 년도는?
>>> Response: 294년입니다. 재임 중 닉슨 대통령의 부통령직은 부통령입니다.承로 출전할 때, 그는 부통령이 아닌 부통령이 되었습니다. 부통령직은 닉슨 대통령은 부통령 재임 중 대통령실장에 임명되었습니다.承은 부통령 재직 기간 중 대통령이 되었습니다. 부통령직은 닉슨 대통령 재임 중 부통령이 되었습니다. 부통령직은 닉슨 대통령과 존슨 대통령의 관계와 밀접한 관련이 있습니다. 부통령직은 닉슨 대통령 재임 중 부통령인 프랭클린 루스벨트 부통령과 부통령입니다. 부통령직은 닉슨 대통령 재임 기간 동안 부통령과 부통령 간에 밀접한 관련이 있었다고 볼 수 있습니다. 부통령직은 존슨 대통령의 정치적 변화와 관련하여 중요한 역할을 했습니다. 부통령직은 닉슨 부통령과 존슨 대통령의 관계를 상징하였습니다. 부통령직은 닉슨 대통령의 부통령의 역사적 생애와 인생과 밀접한 관련이 있었다고 할 수 있습니다. 부통령직은 닉슨 대통령이 부통령직을 수행한 해에 부통령이 되었습니다. 부통령직은 닉슨 대통령이 킹이라는 인물을 부통령으로 선출하였습니다. 부통령직은 닉

2026-03-15:22:29:20 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-15:22:29:27 INFO     [_cli.run:376] Selected Tasks: ['kobest_copa', 'kobest_hellaswag', 'kobest_boolq']
2026-03-15:22:29:29 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-03-15:22:29:29 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/models/skt-kogpt2-base-v2_baseline_output_1_SFT', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}
2026-03-15:22:29:41 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-03-15:22:29:44 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100%|██████████| 149/149 [00:03<00:00, 38.95it/s]
The tied weights 

Passed argument batch_size = auto:1. Detecting largest batch size
Determined largest batch size: 64
hf ({'pretrained': '/home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/models/skt-kogpt2-base-v2_baseline_output_1_SFT', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}), gen_kwargs: ({}), limit: 500.0, num_fewshot: None, batch_size: auto (64)
|     Tasks      |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|----------------|------:|------|-----:|--------|---|-----:|---|------|
|kobest_boolq    |      1|none  |     0|acc     |↑  |0.5340|±  |0.0223|
|                |       |none  |     0|f1      |↑  |0.3481|±  |   N/A|
|kobest_copa     |      1|none  |     0|acc     |↑  |0.4880|±  |0.0224|
|                |       |none  |     0|f1      |↑  |0.4875|±  |   N/A|
|kobest_hellaswag|      1|none  |     0|acc     |↑  |0.2340|±  |0.0190|
|                |       |none  |     0|acc_norm|↑  |0.2660|±  |0.0198|
|                |       |none  |     0|f1      |↑  |0.2334|±  |   N/A|

🚀

2026-03-15:22:30:29 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-15:22:30:37 INFO     [_cli.run:376] Selected Tasks: ['kobest_copa', 'kobest_hellaswag', 'kobest_boolq']
2026-03-15:22:30:39 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-03-15:22:30:39 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/models/skt-kogpt2-base-v2_baseline_output_3_PPO', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}
2026-03-15:22:30:51 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-03-15:22:30:54 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100%|██████████| 149/149 [00:00<00:00, 776.19it/s] 
The tied weight

Passed argument batch_size = auto:1. Detecting largest batch size
Determined largest batch size: 64
hf ({'pretrained': '/home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/models/skt-kogpt2-base-v2_baseline_output_3_PPO', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}), gen_kwargs: ({}), limit: 500.0, num_fewshot: None, batch_size: auto (64)
|     Tasks      |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|----------------|------:|------|-----:|--------|---|-----:|---|------|
|kobest_boolq    |      1|none  |     0|acc     |↑  |0.5340|±  |0.0223|
|                |       |none  |     0|f1      |↑  |0.3481|±  |   N/A|
|kobest_copa     |      1|none  |     0|acc     |↑  |0.4840|±  |0.0224|
|                |       |none  |     0|f1      |↑  |0.4835|±  |   N/A|
|kobest_hellaswag|      1|none  |     0|acc     |↑  |0.2300|±  |0.0188|
|                |       |none  |     0|acc_norm|↑  |0.2700|±  |0.0199|
|                |       |none  |     0|f1      |↑  |0.2291|±  |   N/A|



In [4]:
    """ SFT 관련 param 조정 """

    model_name = "skt/kogpt2-base-v2"
    tcname = model_name.replace("/", "-") + "_case1"

    try:
        root_path = os.path.dirname(os.path.abspath(__file__))
    except:
        root_path = os.getcwd()
        
    cfg = {
        "model_name": model_name,
        "sft_tokenizer": model_name,
        "device": torch.device("xpu" if torch.xpu.is_available() else "cuda" if torch.cuda.is_available() else "cpu"),
        "root_path": root_path,

        "sft_output_dir": os.path.join(root_path, "test", tcname),
        "sft_saved_dir": os.path.join(root_path, "models", tcname + "_output_1_SFT"),
        "rm_saved_dir": os.path.join(root_path, "models", tcname + "_output_2_RM"),
        "ppo_saved_dir": os.path.join(root_path, "models", tcname + "_output_3_PPO"),

        "data_path_1_SFT": os.path.join(root_path, "KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl"),
        "data_path_2_RM": os.path.join(root_path, "KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl"),
        "data_path_3_PPO": os.path.join(root_path, "KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl"),

        "sft_num_train_epochs": 3,              # 1 -> 3
        "sft_per_device_train_batch_size": 8,   # 4 -> 8 : batch size 늘리면 효과는 좋음. ->  gradient가 안정적으로 계산되기 때문
        "sft_per_device_eval_batch_size": 8,    # 4 -> 8 : eval batch size는 메모리 사용량에 영향이 없음. 왜냐하면 eval은 학습이 아니기 때문 ?
        "sft_warmup_steps": 10,                 # 5 -> 10 : 
        "sft_prediction_loss_only": True,
        "sft_fp16": False,

        "rm_batch_size": 4,
        "rm_max_epochs": 1,

        "rm_seed": 230319, 
        "rm_max_train_samples": 1000, 
        "rm_max_len": 512, 
        "rm_lr": 5e-5 
    }

    model = AutoModelForCausalLM.from_pretrained(cfg["model_name"]).to(cfg["device"])
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        cfg["model_name"],
        bos_token='</s>', eos_token='</s>', unk_token='<unk>', pad_token='<pad>', mask_token='<mask>',
        padding_side="right",
        model_max_length=512,
    )

    show_info(cfg)
    run_sft(cfg, model, tokenizer)
    run_reward_model(cfg, tokenizer)
    run_ppo(cfg, model, tokenizer)
    run_evaluation(cfg)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



-------------------- show_info() --------------------
Torch version: 2.7.1+cu118
transformers version: 5.3.0
model_name: skt/kogpt2-base-v2
sft_tokenizer: skt/kogpt2-base-v2
device: cuda
root_path: /home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05
sft_output_dir: /home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/test/skt-kogpt2-base-v2_case1
sft_saved_dir: /home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/models/skt-kogpt2-base-v2_case1_output_1_SFT
rm_saved_dir: /home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/models/skt-kogpt2-base-v2_case1_output_2_RM
ppo_saved_dir: /home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/models/skt-kogpt2-base-v2_case1_output_3_PPO
data_path_1_SFT: /home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl
data_path_2_RM: /home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl
data_path_3_PPO: /home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/KoChatGPT/data_kochatgpt/kochatgpt

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

>>> PPO training skipped as pre-trained model was loaded.

-------------------- generate_custom() --------------------
>>> Instruction:
불고기용 고기 한우에요?
>>> Response:\n\n"불고기용 한우에"가 어떤 고기인지 알려주세요.\n\n> "불고기용 한우에요"라는 말은 식당 이름이나 이름 등이 포함된 단어입니다. 즉, 고기 또는 육류를 먹을 때 사용하는 말입니다.\n\n> "불고기용 한우에요"란 말은 식당 이름이나 음식점이름을 의미합니다. 따라서, 식당 이름이나 음식점에 따라 다를 수 있습니다.\n\n하지만, "불고기용 한우는 한우에요"라는 말이 식당 이름이라면, 이것은 한우의 종류 혹은 맛이 많이 다른 고기를 의미할 수 있습니다. 따라서 한우는 한우를 좋아하지 않는 경우도 있지만, 한우를 좋아하면 한우를 즐겨도 좋다고 할 수 있습니다.美國際在 官吏官)의 "불고기용 한우에요"도 맛있게 먹을 수 있는 음식이 될 것입니다.美國際在 官吏官)에서 "불고기용 한우에요"라는 말이 맛있는 음식이라면 한우는 한류를 좋아하지 않는 것도 좋다고 할 수 있습니다.美國際在 官吏官)이 "불고기용 한우에요"라는 말을 어떻게 요리했는지 알려주시면 더욱 정확한
>>> Instruction:
리처드 닉슨이 43대 부통령직을 수행한 년도는?
>>> Response: 294년입니다.尤無書題: ンリッキン (Takepe of English: 劉備郎) 경질되었다가 복귀한 후 다시 복귀한 것입니다. 경질되었다가 복귀한 후 복귀한 것은 아니며, 柳備郎이 대통령 시절에 부통령을 역임했었습니다. 경질 후에 복귀한 것은 아니며, 柳備郎이 대통령 시절에 부통령을 맡았던 시기에는 대통령 선거와 관련하여 부통령의 선거 개입 논란으로 논란이 있었습니다. 경질 후에는 부통령을 역임하며 새로운 인연을 찾았습니다. 경질 후에는 부통령직을 되찾아 복귀한 적이 없는 것이 일반적입니다. 경질 후에는 부통령직을 

2026-03-15:22:38:15 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-15:22:38:22 INFO     [_cli.run:376] Selected Tasks: ['kobest_copa', 'kobest_hellaswag', 'kobest_boolq']
2026-03-15:22:38:25 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-03-15:22:38:25 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/models/skt-kogpt2-base-v2_case1_output_1_SFT', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}
2026-03-15:22:38:37 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-03-15:22:38:39 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100%|██████████| 149/149 [00:03<00:00, 38.77it/s]
The tied weights map

Passed argument batch_size = auto:1. Detecting largest batch size
Determined largest batch size: 64
hf ({'pretrained': '/home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/models/skt-kogpt2-base-v2_case1_output_1_SFT', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}), gen_kwargs: ({}), limit: 500.0, num_fewshot: None, batch_size: auto (64)
|     Tasks      |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|----------------|------:|------|-----:|--------|---|-----:|---|------|
|kobest_boolq    |      1|none  |     0|acc     |↑  |0.5340|±  |0.0223|
|                |       |none  |     0|f1      |↑  |0.3481|±  |   N/A|
|kobest_copa     |      1|none  |     0|acc     |↑  |0.4840|±  |0.0224|
|                |       |none  |     0|f1      |↑  |0.4826|±  |   N/A|
|kobest_hellaswag|      1|none  |     0|acc     |↑  |0.2380|±  |0.0191|
|                |       |none  |     0|acc_norm|↑  |0.2840|±  |0.0202|
|                |       |none  |     0|f1      |↑  |0.2379|±  |   N/A|

🚀 Ru

2026-03-15:22:39:20 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-15:22:39:29 INFO     [_cli.run:376] Selected Tasks: ['kobest_copa', 'kobest_hellaswag', 'kobest_boolq']
2026-03-15:22:39:32 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-03-15:22:39:32 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/models/skt-kogpt2-base-v2_case1_output_3_PPO', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}
2026-03-15:22:39:44 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-03-15:22:39:46 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100%|██████████| 149/149 [00:00<00:00, 845.47it/s] 
The tied weights m

Passed argument batch_size = auto:1. Detecting largest batch size
Determined largest batch size: 64
hf ({'pretrained': '/home/jovyan/work/repo/AIFFEL_quest_eng/NLP/NLP05/models/skt-kogpt2-base-v2_case1_output_3_PPO', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}), gen_kwargs: ({}), limit: 500.0, num_fewshot: None, batch_size: auto (64)
|     Tasks      |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|----------------|------:|------|-----:|--------|---|-----:|---|------|
|kobest_boolq    |      1|none  |     0|acc     |↑  |0.5340|±  |0.0223|
|                |       |none  |     0|f1      |↑  |0.3481|±  |   N/A|
|kobest_copa     |      1|none  |     0|acc     |↑  |0.4880|±  |0.0224|
|                |       |none  |     0|f1      |↑  |0.4856|±  |   N/A|
|kobest_hellaswag|      1|none  |     0|acc     |↑  |0.2360|±  |0.0190|
|                |       |none  |     0|acc_norm|↑  |0.2860|±  |0.0202|
|                |       |none  |     0|f1      |↑  |0.2360|±  |   N/A|

